# Predicting the NCAA Men's Gymnastics Team Champion

**Can you tell who is going to win the national title before the national meet happens?**

Every NCAA men's gymnastics season ends with one team holding the trophy. This project aims to see
if we can predict that outcome from regular-season data alone. And if so, at what point in the
season can we start to form a realistic prediction.

The dataset is every NCAA team-season from 2013 through 2025 (2020 was cancelled by COVID-19),
scraped from the [Road to Nationals](https://www.roadtonationals.com) API. Each team-season is
described by per-apparatus scoring features, and labeled `champion = 1` or `0`. The 2026 season
is then used as a genuine forward test: the prediction was made from pre-championship data,
before the meet occurred.

### The Results

1. **The obvious approach doesn't work.** Picking the team with the highest average total score
   identifies the champion in only **7 of 12** seasons.
2. **One feature is more predictive than any model, though the model is how we found it.**
   A team's NQA (National Qualifying Average) on **parallel bars** identifies the champion in
   **12 of 12** labeled seasons, and again in 2026 for **13 of 13**. That's a single number, not
   a model, and it's the only feature we found that lined up with the winner every time. It came
   out of the regression: parallel bars kept coming back as the heaviest weight, which is what
   prompted testing it on its own.
3. **The trained model gets close, but I wouldn't bet money on it.** A regularized logistic
   regression on 27 features gets both held-out seasons right (2024, 2025). On the 2026 forward
   test, different versions kept flipping between Stanford and Oklahoma for the top spot, and the
   final margins were too tight to call it either way. Both versions did get the top four right,
   though that may say more about the sport than the model, since the same four or five programs
   dominate season after season.

---

## 0. Environment

The notebook caches its fetched data so the API only has to be hit once. On Colab that cache lives
in Drive; locally it is just a file on disk.

In [ ]:
# Colab only. Mounts Drive so fetched data survives between sessions.
# Running locally? Skip this cell. CACHE_PATH below falls back to a local file.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab, caching to a local file.")

## 1. Setup and Data Source

All data comes from the Road to Nationals (RTN) men's gymnastics API. Two endpoints matter:

| Endpoint | What it gives us |
|---|---|
| `dashboard/{year}/{team_id}` | Team metadata, conference ID, and the **official total score** for each meet |
| `teamconsistency/{year}/{team_id}` | Dated **per-apparatus scores** (floor, pommel, rings, vault, p-bars, high bar) |

We need both. `teamconsistency` is the only source of apparatus-level detail, but `dashboard`
carries the officially recognized team total, and as Section 3 shows, those two do not always
agree. Reconciling them is the first real task.

A conference filter restricts us to NCAA programs. RTN also tracks GymACT teams (the club circuit
several programs moved to after their varsity teams were cut), and mixing those in would corrupt
both the labels and the scoring distributions.

In [ ]:
import requests
import pandas as pd
import numpy as np
import pickle
import os
from datetime import datetime

BASE_URL   = "https://www.roadtonationals.com/api/men"
CACHE_PATH = ('/content/drive/MyDrive/ncaa_gymnastics/raw_data.pkl' if IN_COLAB
              else 'ncaa_raw_data.pkl')

# 2020 is absent: the season was cancelled by COVID-19 before the championship.
SEASONS      = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]

# Big Ten, ECAC, MPSF. Anything else is GymACT or unaffiliated.
NCAA_CONFS   = {'1', '2', '4'}

# Discovered by scanning the ID space, see Section 2.
ALL_TEAM_IDS = [1, 2, 3, 6, 12, 13, 20, 22, 23, 24, 28,
                29, 30, 38, 39, 42, 46, 50, 51, 64, 65]

# RTN returns 403 without these.
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'),
    'Referer': 'https://www.roadtonationals.com/'
}

print(f"{len(SEASONS)} seasons, {len(ALL_TEAM_IDS)} candidate team IDs")
print(f"Cache path: {CACHE_PATH}")

### Optional: load from cache

If the data has already been fetched, this cell restores everything and the fetching cells in
Sections 4-5 can be skipped. On a cold run, skip this cell and let the notebook rebuild from the API.

In [ ]:
# Optional fast path
# Restores: raw_data, champions, df, df_norm
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        saved = pickle.load(f)
    raw_data  = saved['raw_data']
    champions = saved['champions']
    df        = saved['df']
    df_norm   = saved['df_norm']
    print(f"Loaded {len(raw_data)} team-seasons | df {df.shape} | df_norm {df_norm.shape}")
else:
    print("No cache found. Run the fetch cells in Sections 4 and 5.")

## 2. Which Teams Count?

`ALL_TEAM_IDS` above was not guessed. RTN assigns each program a numeric ID with no public
directory, so the roster was found by scanning the ID space and keeping every team whose
`conference_id` marked it as NCAA.

Men's gymnastics is a small sport. The scan turns up roughly 15 to 17 varsity programs per season,
which is why the champion class is so rare (12 champions in 185 team-seasons, about 6.5%). That
class imbalance drives several modeling decisions later.

In [ ]:
# Roster discovery: scan the ID space for NCAA programs
# Run once to regenerate ALL_TEAM_IDS. 2013 is used as the reference season
# because it is the earliest in the study window.
def discover_teams(year, id_range=range(1, 120)):
    found = []
    for team_id in id_range:
        try:
            dash = requests.get(f"{BASE_URL}/dashboard/{year}/{team_id}",
                                headers=HEADERS, timeout=30).json()
            ty   = dash.get('ty_info')
            conf = str(ty.get('conference_id', '')) if isinstance(ty, dict) else ''
            if conf in NCAA_CONFS:
                found.append({'team_id': team_id,
                              'team_name': dash['info']['team_name'],
                              'conf': conf})
        except Exception:
            pass
    return found

teams_2013 = discover_teams(2013)
for t in teams_2013:
    print(f"  ID {t['team_id']:>3}: {t['team_name']:<22} (conf {t['conf']})")
print(f"\nFound {len(teams_2013)} NCAA teams in 2013")

## 3. Is the Data Trustworthy?

Before modeling anything, the two endpoints need to be reconciled.

`teamconsistency` gives six apparatus scores per meet. `dashboard` gives one official team total
per meet. In principle the six apparatus scores should sum to the team total. Where they don't,
either the scrape is misaligned or the underlying data is wrong, and either way, features built on
top of it would be garbage.

The check below computes `delta = |sum(apparatus) - dashboard_total|` for every meet and flags
anything above a small tolerance.

In [ ]:
# Integrity check helper
def check_team_season(year, team_id):
    '''Compare summed apparatus scores against the official dashboard total.'''
    dash = requests.get(f"{BASE_URL}/dashboard/{year}/{team_id}",
                        headers=HEADERS, timeout=30).json()
    cons = requests.get(f"{BASE_URL}/teamconsistency/{year}/{team_id}",
                        headers=HEADERS, timeout=30).json()

    # Dashboard → keep only meets that have an official score
    dash_rows = []
    for m in dash['meets']:
        if m['team_score'] is not None:
            dash_rows.append({
                'date_raw' : m['meet_date'],
                'dashboard': float(m['team_score']),
            })

    # teamconsistency → sum the six apparatus per meet
    apps = ['fxs', 'phs', 'srs', 'vts', 'pbs', 'hbs']
    cons_rows = []
    for i, label in enumerate(cons['labels']):
        cons_rows.append({
            'label'    : label,
            'apparatus': sum(float(cons[a][i]) for a in apps),
        })

    # Align on date. Dashboard dates look like "Sat, Jan-18-2025";
    # teamconsistency dates look like "Jan-18-25".
    for r in dash_rows:
        r['key'] = r['date_raw'].split(', ')[-1].replace('-20', '-')
    for r in cons_rows:
        r['key'] = r['label']

    merged = pd.merge(pd.DataFrame(dash_rows), pd.DataFrame(cons_rows), on='key', how='inner')
    merged['delta'] = (merged['apparatus'] - merged['dashboard']).abs().round(3)
    return merged[['key', 'dashboard', 'apparatus', 'delta']]

### A known-bad example

Illinois 2025 contains a meet whose displayed total is known to be wrong. If the check is working,
it should surface as a non-zero delta.

In [ ]:
il_2025 = check_team_season(2025, 12)
print(il_2025.to_string(index=False))

### First pass: check every NCAA team-season

Now the same check across the whole study window. Note that this cell deliberately uses its own
local variable names (`chk_seasons`, `chk_team_ids`, `chk`) rather than reassigning the global
`SEASONS` / `ALL_TEAM_IDS` / `df`. In the original working notebook these were overwritten here,
which silently truncated the dataset when the notebook was run top-to-bottom.

In [ ]:
# First pass: no conference filter
chk_seasons  = [2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
chk_team_ids = [1, 2, 3, 6, 12, 13, 20, 22, 23, 24, 28, 29, 30, 38, 39, 42, 51]

all_deltas = []
for year in chk_seasons:
    print(f"Checking {year}...", end=" ")
    for team_id in chk_team_ids:
        try:
            chk = check_team_season(year, team_id)
            chk['year'], chk['team_id'] = year, team_id
            all_deltas.append(chk)
        except Exception:
            pass          # program not active that season, or endpoint missing
    print("done")

deltas_raw = pd.concat(all_deltas, ignore_index=True)
flagged    = deltas_raw[deltas_raw['delta'] > 0.05]
print(f"\nTotal meets checked : {len(deltas_raw)}")
print(f"Flagged (delta>0.05): {len(flagged)}  ({len(flagged)/len(deltas_raw):.1%})")

### What the flags actually were

The first pass flagged **102 of 1,539 meets (6.6%)**, too many to ignore. Investigating them
turned up two distinct causes, and only one of them is a real problem:

**1. GymACT contamination.** Several programs (Arizona State, Iowa, Minnesota, Temple) were cut as
varsity teams and continued competing in GymACT. RTN keeps tracking them under the same team ID,
but their `teamconsistency` records in those seasons are garbled, producing deltas of 60-75 points.
These are not scoring errors. They are *invalid comparisons*. Those team-seasons are not NCAA
competition and do not belong in the dataset at all.

**2. Legitimate team neutral deductions.** A cluster of exactly-1.0 deltas turned out to be real.
Neutral deductions (athlete replacement, equipment violations, attire) are applied to the team
total *after* apparatus scoring. The `dashboard` value correctly reflects the penalized final
score; `teamconsistency` shows the raw apparatus sum. Both numbers are right, they just measure
different things.

The fix is to filter on `conference_id` so only genuine NCAA team-seasons are compared.

In [ ]:
# Second pass: NCAA conferences only
clean_deltas, skipped = [], 0

for year in SEASONS:
    print(f"Checking {year}...", end=" ")
    for team_id in ALL_TEAM_IDS:
        try:
            dash = requests.get(f"{BASE_URL}/dashboard/{year}/{team_id}",
                                headers=HEADERS, timeout=30).json()
            ty   = dash.get('ty_info')
            conf = str(ty.get('conference_id', '')) if isinstance(ty, dict) else ''

            if conf not in NCAA_CONFS:      # GymACT or unaffiliated, not comparable
                skipped += 1
                continue

            chk = check_team_season(year, team_id)
            chk['year']      = year
            chk['team_id']   = team_id
            chk['team_name'] = dash['info']['team_name']
            clean_deltas.append(chk)
        except Exception:
            pass
    print("done")

deltas = pd.concat(clean_deltas, ignore_index=True)
flagged_clean = deltas[deltas['delta'] > 0.05]

print(f"\nSkipped (non-NCAA)  : {skipped} team-seasons")
print(f"Total meets checked : {len(deltas)}")
print(f"Flagged (delta>0.05): {len(flagged_clean)}  ({len(flagged_clean)/len(deltas):.1%})")
print("\nFlagged rows by delta value:")
print(flagged_clean['delta'].value_counts().to_string())

### Integrity conclusion

Filtering to NCAA-only team-seasons dropped the flag rate from **6.6% (102/1,539)** to
**1.6% (22/1,415)**, and all 22 survivors are explained:

- **20 rows with delta = 1.0**: confirmed team neutral deductions. The dashboard correctly applies
  the penalty; `teamconsistency` reports the raw apparatus sum. Both are accurate.
- **2 rows with delta = 2.0**: Illinois (Jan-18-2025) and Greenville (Feb-01-2025). Consistent
  with two stacked neutral deductions, though not independently confirmed.

**Verdict: the apparatus data is sound.** Because every feature in this project is built from
per-apparatus scores rather than team totals, the 1.0-point neutral deductions do not propagate
into the feature set. The conference filter established here is reused for the actual data pull.

## 4. Ground Truth: Who Actually Won

The label. `finalresults/{year}` returns the final NCAA Championship standings; the top row is the
champion. This is the target variable: `champion = 1` for the winning team-season, `0` for everyone
else.

Two things worth noticing in the output:

- **Only three programs win.** Michigan, Oklahoma, and Stanford account for every title from 2013
  to 2026. Any model will inherit that concentration, and it makes "predict a dynasty" a genuinely
  hard baseline to beat.
- **Score levels are not comparable across seasons.** Champions score ~443 in 2013 and ~332 in
  2025. That is a rule change, not a collapse in quality, and Section 7 deals with it.

In [ ]:
# Fetch champions for every labeled season
champions = {}
for year in SEASONS:
    try:
        data = requests.get(f"{BASE_URL}/finalresults/{year}",
                            headers=HEADERS, timeout=30).json()
        if data.get('data'):
            champ = data['data'][0]
            champions[year] = {
                'team_id'    : int(champ['team_id']),
                'team_name'  : champ['team_name'],
                'final_score': float(champ['ncaa_final']),
            }
            print(f"{year}: {champ['team_name']:<12} ({champ['ncaa_final']})")
    except Exception:
        print(f"{year}: no results available")

print(f"\n{len(champions)} labeled seasons")
print(f"Distinct champions: {sorted({c['team_name'] for c in champions.values()})}")

## 5. Building the Dataset

One row per NCAA team-season. Each record keeps the dated per-apparatus scores (the raw material
for features), the official meet totals, and the champion label.

The conference filter from Section 3 is applied here, so GymACT seasons never enter the dataset.

In [ ]:
# Fetch all raw data
raw_data = []
for year in SEASONS:
    print(f"Fetching {year}...", end=" ")
    count = 0
    for team_id in ALL_TEAM_IDS:
        try:
            dash = requests.get(f"{BASE_URL}/dashboard/{year}/{team_id}",
                                headers=HEADERS, timeout=30).json()
            ty   = dash.get('ty_info')
            conf = str(ty.get('conference_id', '')) if isinstance(ty, dict) else ''
            if conf not in NCAA_CONFS:
                continue

            meet_scores = [float(m['team_score']) for m in dash['meets']
                           if m['team_score'] is not None]
            if not meet_scores:
                continue

            cons = requests.get(f"{BASE_URL}/teamconsistency/{year}/{team_id}",
                                headers=HEADERS, timeout=30).json()

            raw_data.append({
                'year'       : year,
                'team_id'    : team_id,
                'team_name'  : dash['info']['team_name'],
                'champion'   : int(champions.get(year, {}).get('team_id') == team_id),
                'meet_scores': meet_scores,
                'meet_dates' : cons['labels'],
                'apparatus'  : {a: [float(x) for x in cons[a]]
                                for a in ['fxs', 'phs', 'srs', 'vts', 'pbs', 'hbs']},
            })
            count += 1
        except Exception:
            pass
    print(f"{count} teams")

print(f"\nTotal team-seasons : {len(raw_data)}")
print(f"Champion entries   : {sum(r['champion'] for r in raw_data)}")

## 6. Feature Engineering

This is where the project's actual contribution lives. Two decisions matter more than everything
else in the notebook combined.

### Decision 1: the April 8 cutoff

A model that predicts the championship using championship data has learned nothing. Every feature
is therefore computed from meets on or before **April 8**, which sits in the gap between the
conference championships and the NCAA Championship. Conference meets land before it; national
semifinals and finals land after it.

This is the single assumption the whole framing rests on. It is worth re-verifying against the meet
labels rather than trusting it. We picked the date by inspecting where conference meets fell, and
a season where a conference meet ran late would quietly leak.

> **TODO (Alex):** confirm from the meet labels that April 8 cleanly separates conference from
> national meets in *all* 13 seasons, and note any exception here.

### Decision 2: NQA instead of a season average

A plain season average is a bad summary of a gymnastics team. Scores are volatile, a single
disastrous meet can drag the mean down, and the sport itself does not rank teams that way.

NCAA gymnastics uses the **National Qualifying Average (NQA)**, and the version implemented here
is: *drop the highest regular-season score, average the next three, then average that against the
conference championship score.* Dropping the top score removes the outlier peak; weighting the
conference result into the average emphasizes late-season form, when a team is closest to the
condition it will be in at nationals.

Swapping per-event means for per-event NQA is what made the project work. The parallel-bars
baseline went from 10/12 to 12/12 on that change alone (Section 9).

### The feature set

For each of the six apparatus (`fx`, `ph`, `sr`, `vt`, `pb`, `hb`):

| Feature | Meaning |
|---|---|
| `nqa_*`   | National Qualifying Average, our main strength measure |
| `std_*`   | Standard deviation across meets, i.e. consistency |
| `max_*`   | Season best, i.e. ceiling |
| `slope_*` | Linear trend over the last four meets, i.e. late-season trajectory |

Plus season-level `mean_total`, `max_total`, `slope_total`, `conf_champ_total`, and `n_meets`.
That is 29 columns before any are dropped.

In [ ]:
# Feature computation
def parse_meet_date(label, year):
    '''Convert a teamconsistency label like "Jan-18-25" to a datetime.'''
    try:
        return datetime.strptime(label, '%b-%d-%y')
    except Exception:
        return None


def compute_features(record, cutoff_month=4, cutoff_day=8, min_meets=2):
    '''
    Compute features for one team-season using only meets on or before the cutoff.

    Default cutoff is April 8, after the conference championships and before
    the NCAA Championship, so no feature can encode the outcome being predicted.
    '''
    apps  = ['fxs', 'phs', 'srs', 'vts', 'pbs', 'hbs']
    names = ['fx',  'ph',  'sr',  'vt',  'pb',  'hb']
    year  = record['year']

    # --- Restrict to meets before the cutoff ---
    cutoff = datetime(year, cutoff_month, cutoff_day)
    valid  = []
    for i, label in enumerate(record['meet_dates']):
        dt = parse_meet_date(label, year)
        if dt is not None and dt <= cutoff:
            valid.append(i)

    if len(valid) < min_meets:
        return None                      # too few meets to summarize

    feats = {}
    for app, name in zip(apps, names):
        all_scores = np.array(record['apparatus'][app])
        scores     = all_scores[valid]

        # NQA: drop the highest regular-season score, take the next three,
        # then average those against the conference championship score.
        conf_score = all_scores[valid[-1]]     # last meet before cutoff
        reg_scores = scores[:-1]               # everything except conference
        sorted_reg = np.sort(reg_scores)[::-1]
        top3_reg   = sorted_reg[1:4]           # drop the peak, keep next three
        feats[f'nqa_{name}'] = (np.sum(top3_reg) + conf_score) / 4

        feats[f'std_{name}'] = np.std(scores)
        feats[f'max_{name}'] = np.max(scores)

        last = scores[-4:]                     # late-season trajectory
        feats[f'slope_{name}'] = np.polyfit(range(len(last)), last, 1)[0]

    # --- Season-level features ---
    total_valid = np.array(record['meet_scores'])[:len(valid)]
    feats['mean_total']       = float(np.mean(total_valid))
    feats['max_total']        = float(np.max(total_valid))
    last4_total               = total_valid[-4:]
    feats['slope_total']      = float(np.polyfit(range(len(last4_total)), last4_total, 1)[0])
    feats['conf_champ_total'] = float(total_valid[-1])
    feats['n_meets']          = len(valid)

    return feats

### Verifying the calculation by hand

Automated feature code is easy to get subtly wrong, so the next two cells check it against numbers
computed by eye. The first prints Illinois 2025's raw pommel scores alongside the derived features.
The second re-derives NQA for two teams using the formula written out longhand.

Note that in the original notebook this second check used a *different* formula than
`compute_features`. It took the top four of all meets including conference, rather than dropping
the highest regular-season score and appending conference separately. Those agree only when the
conference score is not among the top four, so it was not actually verifying the function. The
version below mirrors `compute_features` exactly.

In [ ]:
# Hand check 1: Illinois 2025 pommel horse
il = next(r for r in raw_data if r['team_id'] == 12 and r['year'] == 2025)
il_feats = compute_features(il)

cutoff = datetime(2025, 4, 8)
valid  = [i for i, lab in enumerate(il['meet_dates'])
          if parse_meet_date(lab, 2025) and parse_meet_date(lab, 2025) <= cutoff]

print("Illinois 2025 pommel horse scores on or before Apr 8:")
for i in valid:
    print(f"  {il['meet_dates'][i]}: {il['apparatus']['phs'][i]:.3f}")

print(f"\nComputed features:")
print(f"  nqa_ph   : {il_feats['nqa_ph']:.4f}")
print(f"  std_ph   : {il_feats['std_ph']:.4f}")
print(f"  max_ph   : {il_feats['max_ph']:.4f}   <- should match the largest score above")
print(f"  slope_ph : {il_feats['slope_ph']:+.4f}  <- trend over the last four meets")
print(f"  n_meets  : {il_feats['n_meets']}")

In [ ]:
# Hand check 2: re-derive NQA longhand, mirroring compute_features
for record in raw_data:
    if record['year'] == 2024 and record['team_name'] in ('Stanford', 'Michigan'):
        cutoff = datetime(2024, 4, 8)
        valid  = [i for i, lab in enumerate(record['meet_dates'])
                  if parse_meet_date(lab, 2024) and parse_meet_date(lab, 2024) <= cutoff]

        all_pb     = np.array(record['apparatus']['pbs'])
        scores     = all_pb[valid]
        conf_score = all_pb[valid[-1]]
        sorted_reg = np.sort(scores[:-1])[::-1]
        top3_reg   = sorted_reg[1:4]
        nqa_manual = (np.sum(top3_reg) + conf_score) / 4

        nqa_fn = compute_features(record)['nqa_pb']

        print(f"\n{record['team_name']} 2024 parallel bars")
        print(f"  scores before cutoff : {scores.round(3)}")
        print(f"  conference score     : {conf_score:.3f}")
        print(f"  top 3 after dropping peak: {top3_reg.round(3)}")
        print(f"  NQA by hand          : {nqa_manual:.4f}")
        print(f"  NQA from function    : {nqa_fn:.4f}")
        print(f"  match                : {np.isclose(nqa_manual, nqa_fn)}")

### Assembling the feature table

In [ ]:
# Build the feature table
rows = []
for record in raw_data:
    feats = compute_features(record)
    if feats is None:                       # too few meets before the cutoff
        continue
    feats['year']     = record['year']
    feats['team']     = record['team_name']
    feats['champion'] = record['champion']
    rows.append(feats)

df   = pd.DataFrame(rows)
META = ['year', 'team', 'champion']
df   = df[META + [c for c in df.columns if c not in META]]

print(f"Shape     : {df.shape}")
print(f"Champions : {df['champion'].sum()}  ({df['champion'].mean():.1%} of rows)")
print(f"\n{len(df.columns) - len(META)} feature columns:")
print([c for c in df.columns if c not in META])

## 7. Era Normalization

Raw scores are not comparable across seasons. The 2024 rule change (counting four scores instead of
five, eight skills instead of ten) dropped team totals by roughly 80-100 points overnight, and
smaller changes (vault difficulty tables, element group requirements, execution deduction
standards) happen almost every year. On top of that, the overall talent level drifts.

A model trained on raw values would learn "high absolute score means champion" and then fail
completely the moment the scoring regime shifts.

**Fix: within-season z-scoring.** Every feature is converted to
`(value - season_mean) / season_std`, computed *within* each season.

- **What this keeps:** how far above or below its contemporaries a team was. That is what actually
  predicts a title.
- **What this erases:** absolute score level, which is meaningless across rule changes.

A useful side effect is that this makes each season self-contained, which is what lets the 2026
season be normalized on its own and scored by a model trained on earlier eras.

In [ ]:
# Within-season z-scoring
raw_feature_cols = [c for c in df.columns if c not in META]

df_norm = df.copy()
for col in raw_feature_cols:
    season_mean = df.groupby('year')[col].transform('mean')
    season_std  = df.groupby('year')[col].transform('std')
    df_norm[col] = (df[col] - season_mean) / season_std

# Sanity check: every feature should now average ~0 within every season
print("Per-season means after z-scoring (should all be ~0):")
print(df_norm.groupby('year')[raw_feature_cols[:4]].mean().round(4).to_string())

print(f"\nShape     : {df_norm.shape}")
print(f"Champions : {int(df_norm['champion'].sum())}")

champ_example = df_norm[df_norm['champion'] == 1].iloc[-1]
print(f"\nMost recent champion, z-scored: {champ_example['team']} {int(champ_example['year'])}:")
print(champ_example[raw_feature_cols].astype(float).round(3).to_string())

In [ ]:
# Cache everything so the API only gets hit once
cache_dir = os.path.dirname(CACHE_PATH)
if cache_dir:
    os.makedirs(cache_dir, exist_ok=True)
with open(CACHE_PATH, 'wb') as f:
    pickle.dump({'raw_data': raw_data, 'champions': champions,
                 'df': df, 'df_norm': df_norm}, f)
print(f"Cached to {CACHE_PATH}")

## 8. Train / Test Split

The split is **temporal, not random**. Training uses 2013-2023; 2024 and 2025 are held out entirely.

Random splitting would be indefensible here. Team-seasons within a season are not independent. They
are z-scored against each other, so a randomly held-out row would have helped define the mean and
standard deviation used to normalize its own training-set neighbors. A temporal split also mirrors
the actual use case: predict a season you have never seen.

### Which features the model uses

Two of the 29 columns are dropped:

- **`std_vt`**: vault standard deviation was adding noise rather than signal. Because higher
  variance was being rewarded, the feature effectively credited teams for being *inconsistent* on
  vault, which is backwards. It also picked up an outsized weight at `C=1.0`, a classic overfitting
  symptom.
- **`conf_champ_total`**: dropped from the final model. See the note below.

> **TODO (Alex): confirm this.** The write-up describes the final model as 27 features
> *including* the conference championship total, but 29 columns minus `std_vt` is 28, and the
> reported 2026 probabilities (Oklahoma 0.870, Stanford 0.827, Michigan 0.688, Nebraska 0.255)
> match the variant with `conf_champ_total` **removed**, to six decimal places. The notebook is
> built that way here. Section 12 keeps the alternative as a labeled ablation so the difference is
> visible either way.

Dropping it is also defensible on its own terms: `conf_champ_total` is the last observed team
performance before the target event, and it carried the third-largest weight in the model. That is
not leakage, since it falls inside the April 8 cutoff, but it is close enough to the outcome to be
worth removing rather than defending.

`n_meets` is retained but deliberately **not interpreted**. In this sport, meet count is not a
strength signal: strong programs sometimes compete less to stay fresh for the postseason. Its
weight is small and negative, and no claim is made about what that means.

In [ ]:
# Feature selection (defined ONCE, never reassigned)
# std_vt          : rewarded inconsistency on vault; noise, not signal
# conf_champ_total: last performance before the target event; dropped from the final model
DROPPED = ['std_vt', 'conf_champ_total']
feature_cols = [c for c in df_norm.columns if c not in META + DROPPED]

train = df_norm[df_norm['year'] <= 2023]
test  = df_norm[df_norm['year'] >= 2024]

X_train, y_train = train[feature_cols], train['champion']
X_test,  y_test  = test[feature_cols],  test['champion']

print(f"Features : {len(feature_cols)}  (dropped {DROPPED})")
print(f"Train    : {X_train.shape[0]} rows, {int(y_train.sum())} champions "
      f"({train['year'].min()}-{train['year'].max()})")
print(f"Test     : {X_test.shape[0]} rows, {int(y_test.sum())} champions "
      f"({test['year'].min()}-{test['year'].max()})")

print("\nTeams per season:")
print(df_norm.groupby('year').size().to_string())

## 9. Baselines: How Hard Is This Actually?

Before fitting anything, establish what a model has to beat. Each baseline is a single rule: in each
season, pick the team that leads some one number, and check whether that team won the title.

This section is where the project's real finding lives, so it comes before the model rather than
after it.

In [ ]:
# Single-number baselines, evaluated on every labeled season
def pb_variants(record, cutoff_month=4, cutoff_day=8, min_meets=2):
    '''Three ways of summarizing a team's parallel-bars season.'''
    year   = record['year']
    cutoff = datetime(year, cutoff_month, cutoff_day)
    valid  = [i for i, lab in enumerate(record['meet_dates'])
              if parse_meet_date(lab, year) and parse_meet_date(lab, year) <= cutoff]
    if len(valid) < min_meets:
        return None

    all_pb     = np.array(record['apparatus']['pbs'])
    scores     = all_pb[valid]
    conf_score = all_pb[valid[-1]]
    sorted_reg = np.sort(scores[:-1])[::-1]

    return {
        'mean_pb': float(np.mean(scores)),                          # plain average
        'max_pb' : float(np.max(scores)),                           # season best
        'top4_pb': float(np.mean(np.sort(scores)[::-1][:4])),       # average of top four
        'nqa_pb' : float((np.sum(sorted_reg[1:4]) + conf_score) / 4),  # NQA
    }


pb_rows = []
for record in raw_data:
    v = pb_variants(record)
    if v is None:
        continue
    v.update(year=record['year'], team=record['team_name'], champion=record['champion'])
    pb_rows.append(v)

pb_df = pd.DataFrame(pb_rows).merge(
    df[['year', 'team', 'mean_total']], on=['year', 'team'], how='left')

BASELINES = ['mean_total', 'mean_pb', 'max_pb', 'top4_pb', 'nqa_pb']

print(f"{'Season':<8}{'Champion':<12}" + "".join(f"{b:<16}" for b in BASELINES))
print("-" * (20 + 16 * len(BASELINES)))

hits = {b: 0 for b in BASELINES}
for yr, sub in pb_df.groupby('year'):
    champ = sub.loc[sub['champion'] == 1, 'team'].iloc[0]
    line  = f"{yr:<8}{champ:<12}"
    for b in BASELINES:
        leader = sub.loc[sub[b].idxmax(), 'team']
        ok     = leader == champ
        hits[b] += ok
        line += f"{leader + (' OK' if ok else ' X'):<16}"
    print(line)

n = pb_df['year'].nunique()
print("-" * (20 + 16 * len(BASELINES)))
print(f"{'ACCURACY':<20}" + "".join(f"{str(hits[b]) + '/' + str(n):<16}" for b in BASELINES))

### Reading this table

The progression is the whole story:

| Baseline | Accuracy | What it says |
|---|---|---|
| Highest `mean_total` | **7/12** | The intuitive answer, "best team all season wins", is wrong a third of the time |
| Highest `mean_pb` | ~10/12 | Parallel bars alone already beats whole-team average scoring |
| Highest `max_pb` | **10/12** | Ceiling on p-bars |
| Highest `top4_pb` | **10/12** | Average of the four best p-bars meets |
| **Highest `nqa_pb`** | **12/12** | Perfect on every labeled season |

Two things follow.

**Parallel bars carry unusual signal.** Averaging across all six apparatus (`mean_total`) is *worse*
than looking at one apparatus. The plausible reading is that p-bars is the highest-variance,
highest-difficulty event in the men's format, so it separates elite teams from merely good ones more
sharply than events where the field is bunched.

**The NQA construction is doing real work.** The same apparatus summarized three other ways gets
10/12; summarized as NQA it gets 12/12. Dropping the peak score and weighting the conference result
is what converts a good signal into a perfect one on this sample.

**A note on ordering.** We show the baselines before the model, because it reads better to know
what the bar is before watching something try to clear it. That is not the order we found them in.
Parallel bars first turned up as the heaviest weight in the regression (Section 11), and that is
what sent us back to test it on its own. The model was the search tool, not the product.

**The honest caveat:** 12 seasons with 3 distinct champions is a very small sample, and a rule we
picked *after* looking at those seasons is going to flatter itself. We read 12/12 as "this feature
deserves to be the backbone," not as a 100% accurate predictor. The 2026 forward test in Section 14
is the only genuinely out-of-sample evidence we have, and `nqa_pb` passes it, going 13/13.

This is also why we considered a binary `is_nqa_pb_leader` indicator and then threw it out. On this
sample it would be a perfect predictor by construction, which is just overfitting with extra steps.

## 10. Logistic Regression

With a 12/12 baseline on the board, the question for any model is whether it adds anything.

Two settings deserve explanation:

- **`class_weight='balanced'`**: champions are 6.5% of rows. Without upweighting the rare class, a
  model can predict "not champion" every time, score 93.5% accuracy, and be useless. This is also
  why accuracy is never used as the metric here; what matters is whether the champion gets the
  highest probability *within its own season*.
- **`C=0.1`**: regularization strength, chosen from the sweep in Section 12. `C=1.0` overfits
  (individual weights blow up, `std_vt` in particular); `C=0.01` squeezes every probability into a
  band too narrow to separate contenders from filler.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(C=0.1, class_weight='balanced',
                           max_iter=1000, solver='lbfgs', random_state=42)
model.fit(X_train, y_train)

results = test[['year', 'team', 'champion']].copy()
results['prob_champion'] = model.predict_proba(X_test)[:, 1]

for yr in sorted(results['year'].unique()):
    print(f"\n=== {yr} ===")
    print(results[results['year'] == yr]
          .sort_values('prob_champion', ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\n" + "=" * 60)
for yr in sorted(results['year'].unique()):
    r      = results[results['year'] == yr]
    top    = r.loc[r['prob_champion'].idxmax()]
    actual = r.loc[r['champion'] == 1].iloc[0]
    ok     = top['team'] == actual['team']
    print(f"{yr}: picked {top['team']:<12} ({top['prob_champion']:.3f})  "
          f"actual {actual['team']:<12} {'CORRECT' if ok else 'WRONG'}")

Both held-out seasons correct. That is 2 for 2, on a two-season test set, so it is not much to go
on by itself. The more useful thing this model gives us is what it leaned on, which is the next
section.

## 11. What Did the Model Learn?

Because the features are z-scored, the coefficients are directly comparable: a larger weight means
that feature moves the champion probability more.

The second half of this cell asks a sharper question. For every season including the ones the model
trained on, what rank did the *actual* champion receive? Seasons where the champion ranks below
first are the upsets, and they are where the model's assumptions break.

In [ ]:
# Coefficients
weights = pd.Series(model.coef_[0], index=feature_cols).sort_values(ascending=False)

print("Logistic regression weights (positive pushes toward champion)\n")
for feat, w in weights.items():
    bar = '#' * int(abs(w) * 12)
    print(f"  {feat:>16s}  {w:+.3f}  {bar}")
print(f"\n  intercept: {model.intercept_[0]:+.3f}")

# Where did each actual champion rank?
all_results = df_norm[['year', 'team', 'champion']].copy()
all_results['prob'] = model.predict_proba(df_norm[feature_cols])[:, 1]

print("\n\nRank of the actual champion, by season")
print("(2024-2025 are held out; earlier seasons are in-sample)\n")
for yr in sorted(all_results['year'].unique()):
    season = (all_results[all_results['year'] == yr]
              .sort_values('prob', ascending=False).reset_index(drop=True))
    season['rank'] = range(1, len(season) + 1)
    champ  = season[season['champion'] == 1].iloc[0]
    flag   = "" if champ['rank'] == 1 else "   <- upset"
    held   = " (held out)" if yr >= 2024 else ""
    print(f"  {yr}: {champ['team']:>12}  prob={champ['prob']:.3f}  "
          f"rank {int(champ['rank'])}/{len(season)}{flag}{held}")

This is the cell that started everything. `nqa_pb` and `max_pb` come back as the two heaviest
weights, and that is what sent us off to test parallel bars on its own as a single-number rule. The
baseline table in Section 9 came out of this result, not the other way around, even though we
present it first.

Worth saying plainly: the regression did not beat the simple rule, but we would not have found the
simple rule without it.

## 12. Ablations

Three questions: does the regularization strength matter, do the consistency features earn their
place, and what happens if `conf_champ_total` goes back in?

In [ ]:
# Regularization sweep
print("C      test seasons correct   probability spread (2025)")
print("-" * 56)
for c_val in [1.0, 0.1, 0.01]:
    m = LogisticRegression(C=c_val, class_weight='balanced',
                           max_iter=1000, solver='lbfgs', random_state=42)
    m.fit(X_train, y_train)

    r = test[['year', 'team', 'champion']].copy()
    r['prob'] = m.predict_proba(X_test)[:, 1]

    correct = 0
    for yr in sorted(r['year'].unique()):
        s = r[r['year'] == yr]
        correct += int(s.loc[s['prob'].idxmax(), 'team'] ==
                       s.loc[s['champion'] == 1, 'team'].iloc[0])

    s2025  = r[r['year'] == 2025]['prob']
    spread = s2025.max() - s2025.min()
    print(f"{c_val:<7}{correct}/2                    {spread:.3f}   "
          f"(max weight {np.abs(m.coef_[0]).max():.2f})")

`C=1.0` and `C=0.1` both get the test seasons right, so accuracy alone does not choose between them.
The tiebreaker is weight stability. At `C=1.0` individual coefficients grow large, which is what
originally surfaced the `std_vt` problem. `C=0.01` compresses the probability spread until
contenders and filler teams are hard to tell apart.

In [ ]:
# Do the consistency (std_*) features matter?
no_std = [c for c in feature_cols if not c.startswith('std_')]
print(f"Dropping {len(feature_cols) - len(no_std)} std_ features "
      f"-> {len(no_std)} remaining\n")

m = LogisticRegression(C=0.1, class_weight='balanced',
                       max_iter=1000, solver='lbfgs', random_state=42)
m.fit(X_train[no_std], y_train)

r = test[['year', 'team', 'champion']].copy()
r['prob'] = m.predict_proba(X_test[no_std])[:, 1]
for yr in sorted(r['year'].unique()):
    s      = r[r['year'] == yr]
    top    = s.loc[s['prob'].idxmax()]
    actual = s.loc[s['champion'] == 1, 'team'].iloc[0]
    print(f"  {yr}: picked {top['team']:<12} ({top['prob']:.3f})  "
          f"actual {actual:<12} {'CORRECT' if top['team'] == actual else 'WRONG'}")

Removing the `std_*` group as a block breaks both test seasons. Consistency genuinely matters. The
problem with `std_vt` was specific to vault, not to the idea of measuring variance. This is the
evidence for dropping that one feature rather than the whole family.

In [ ]:
# Putting conf_champ_total back in
with_conf = [c for c in df_norm.columns if c not in META + ['std_vt']]
print(f"Final model : {len(feature_cols)} features (conf_champ_total OUT)")
print(f"Alternative : {len(with_conf)} features (conf_champ_total IN)\n")

m = LogisticRegression(C=0.1, class_weight='balanced',
                       max_iter=1000, solver='lbfgs', random_state=42)
m.fit(train[with_conf], y_train)

r = test[['year', 'team', 'champion']].copy()
r['prob'] = m.predict_proba(test[with_conf])[:, 1]
for yr in sorted(r['year'].unique()):
    s      = r[r['year'] == yr]
    top    = s.loc[s['prob'].idxmax()]
    actual = s.loc[s['champion'] == 1, 'team'].iloc[0]
    print(f"  {yr}: picked {top['team']:<12} ({top['prob']:.3f})  "
          f"actual {actual:<12} {'CORRECT' if top['team'] == actual else 'WRONG'}")

w = pd.Series(m.coef_[0], index=with_conf).sort_values(ascending=False)
print(f"\n  conf_champ_total weight: {w['conf_champ_total']:+.3f} "
      f"(rank {list(w.index).index('conf_champ_total') + 1} of {len(w)})")

## 13. Other Models

### Support vector machine

An RBF-kernel SVM on the same features. It gets the two held-out seasons right, but its ranking of
the rest of the field is not credible. Service academies and small programs surface above
established contenders. A kernel method with 155 training rows and 10 positive examples has enough
freedom to carve out regions that fit the training champions without corresponding to anything real.

It is kept here because the failure is informative: getting the top pick right is not the same as
having learned the structure of the problem.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm.fit(X_train, y_train)

r = test[['year', 'team', 'champion']].copy()
r['prob'] = svm.predict_proba(X_test)[:, 1]

for yr in sorted(r['year'].unique()):
    s      = r[r['year'] == yr]
    top    = s.loc[s['prob'].idxmax()]
    actual = s.loc[s['champion'] == 1, 'team'].iloc[0]
    print(f"{yr}: picked {top['team']:<12} ({top['prob']:.3f})  "
          f"actual {actual:<12} {'CORRECT' if top['team'] == actual else 'WRONG'}")

print("\nFull 2025 ranking, note where the service academies land:")
print(r[r['year'] == 2025].sort_values('prob', ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Is the problem even separable?

Projecting all 185 team-seasons onto their first two principal components shows what any classifier
is up against. Champions sit at the edge of the cloud rather than in a cleanly separable cluster.
They are extreme, not distinct.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_2d = pca.fit_transform(df_norm[feature_cols])
is_champ = df_norm['champion'].values == 1

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_2d[~is_champ, 0], X_2d[~is_champ, 1],
           c='#b0b7c3', alpha=0.65, s=42, label='Non-champion', edgecolor='none')
ax.scatter(X_2d[is_champ, 0], X_2d[is_champ, 1],
           c='#c8102e', s=130, label='Champion', edgecolor='white', linewidth=1.4, zorder=3)

for (x, y), (_, row) in zip(X_2d[is_champ], df_norm[is_champ].iterrows()):
    ax.annotate(f"{row['team']} {int(row['year'])}", (x, y),
                textcoords="offset points", xytext=(8, 4), fontsize=8, color='#444')

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} of variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} of variance)")
ax.set_title("NCAA team-seasons in feature space, 2013-2025")
ax.legend(frameon=False)
ax.grid(alpha=0.25, linewidth=0.6)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 14. The 2026 Forward Test

Everything so far is retrospective. This section is the only genuinely out-of-sample evidence in the
project: the model was trained on all 12 labeled seasons, then used to predict a championship that
**had not yet been contested**.

The procedure:

1. Fetch 2026 regular-season data, applying the same April 8 cutoff so only pre-championship meets
   are used.
2. Z-score within the 2026 season alone. This is exactly why era normalization was built the way it
   was in Section 7.
3. Score with a model retrained on all of 2013-2025.
4. *Then* reveal the actual result.

In [ ]:
# Step 1: fetch 2026, pre-cutoff meets only
raw_2026 = []
for team_id in ALL_TEAM_IDS:
    try:
        dash = requests.get(f"{BASE_URL}/dashboard/2026/{team_id}",
                            headers=HEADERS, timeout=30).json()
        ty   = dash.get('ty_info')
        conf = str(ty.get('conference_id', '')) if isinstance(ty, dict) else ''
        if conf not in NCAA_CONFS:
            continue
        meet_scores = [float(m['team_score']) for m in dash['meets']
                       if m['team_score'] is not None]
        if not meet_scores:
            continue
        cons = requests.get(f"{BASE_URL}/teamconsistency/2026/{team_id}",
                            headers=HEADERS, timeout=30).json()
        raw_2026.append({
            'year': 2026, 'team_id': team_id, 'team_name': dash['info']['team_name'],
            'champion': 0,                       # unknown at prediction time
            'meet_scores': meet_scores, 'meet_dates': cons['labels'],
            'apparatus': {a: [float(x) for x in cons[a]]
                          for a in ['fxs', 'phs', 'srs', 'vts', 'pbs', 'hbs']},
        })
    except Exception:
        pass

rows_2026 = []
for record in raw_2026:
    feats = compute_features(record)
    if feats is None:
        continue
    feats['year'], feats['team'], feats['champion'] = 2026, record['team_name'], 0
    rows_2026.append(feats)

df_2026 = pd.DataFrame(rows_2026)
df_2026 = df_2026[META + [c for c in df_2026.columns if c not in META]]

# Step 2: z-score within 2026 only
df_2026_norm = df_2026.copy()
for col in [c for c in df_2026.columns if c not in META]:
    df_2026_norm[col] = (df_2026[col] - df_2026[col].mean()) / df_2026[col].std()

print(f"2026 teams with enough pre-cutoff meets: {len(df_2026)}")
print(f"Meets used per team: {df_2026['n_meets'].min()}-{df_2026['n_meets'].max()}")

In [ ]:
# Step 3: retrain on all 12 labeled seasons, then predict 2026
model_final = LogisticRegression(C=0.1, class_weight='balanced',
                                 max_iter=1000, solver='lbfgs', random_state=42)
model_final.fit(df_norm[feature_cols], df_norm['champion'])
print(f"Trained on {len(df_norm)} team-seasons, "
      f"{int(df_norm['champion'].sum())} champions, {len(feature_cols)} features\n")

pred_2026 = df_2026_norm[['team']].copy()
pred_2026['prob_champion'] = model_final.predict_proba(df_2026_norm[feature_cols])[:, 1]
pred_2026 = pred_2026.sort_values('prob_champion', ascending=False).reset_index(drop=True)
pred_2026.index += 1

print("PREDICTION, made from pre-championship data\n")
print(pred_2026.to_string(float_format=lambda v: f"{v:.4f}"))

print(f"\nSingle-feature baseline (highest nqa_pb): "
      f"{df_2026.loc[df_2026['nqa_pb'].idxmax(), 'team']}")

In [ ]:
# Step 4: the actual result
final_2026 = requests.get(f"{BASE_URL}/finalresults/2026",
                          headers=HEADERS, timeout=30).json()['data']

actual = pd.DataFrame([{'team': r['team_name'], 'final_score': float(r['ncaa_final'])}
                       for r in final_2026 if r.get('ncaa_final') is not None])
actual.index = range(1, len(actual) + 1)

print("ACTUAL 2026 NCAA Championship result\n")
print(actual.head(6).to_string(float_format=lambda v: f"{v:.3f}"))

# Side-by-side comparison of the top four
print("\n\nPredicted vs actual, top 4\n")
print(f"{'':>4}  {'PREDICTED':<24}{'ACTUAL':<24}")
for i in range(1, 5):
    p = f"{pred_2026.loc[i, 'team']} ({pred_2026.loc[i, 'prob_champion']:.3f})"
    a = f"{actual.loc[i, 'team']} ({actual.loc[i, 'final_score']:.3f})"
    print(f"{i:>4}. {p:<24}{a:<24}")

margin = actual.loc[1, 'final_score'] - actual.loc[2, 'final_score']
print(f"\nWinning margin: {margin:.3f} points")

### Reading the forward test

The model picked all four teams that would finish in the top four, and got 3rd and 4th in the right
order. It ranked Oklahoma first and Stanford second. Stanford won by 1.33 points out of roughly 330.

We are not going to call that a miss, and we are not going to call it a hit either. Across the
versions we trained, the top spot moved between Stanford and Oklahoma depending on which features
were in. A 0.4% margin is not a gap the model is equipped to resolve, and the honest read is that it
narrowed the field to the right four teams and then ran out of resolution.

The top-four result deserves a caveat too. Michigan, Oklahoma, and Stanford have won every title
since 2013, and the contender group barely changes year to year. Identifying it is a lower bar than
it sounds like.

What is worth noting is that the single-feature `nqa_pb` rule said Stanford, which takes it to
13/13.

### Why they disagreed: binary ranking vs. continuous features

This gap is the most interesting thing we got out of the project.

The `nqa_pb` baseline is a **ranking** rule. It only asks who leads, and Stanford led parallel bars
(54.950 to Oklahoma's 54.775). The logistic regression works in **continuous** values, so that
0.175-point edge was a small nudge, and Oklahoma outweighed it by being marginally stronger across
the rest of its profile.

Neither approach is wrong in general. But it does mean the model's extra features diluted the one
signal that mattered most, and with 12 positive examples there is not enough data for it to learn
how much to trust its own best feature.

## 15. When Does the Model Know?

April 8 was chosen because it maximizes information. But how early does the picture stabilize?

Re-running the entire pipeline at successively later cutoffs answers that. Features that require a
conference championship (`nqa_*`, `conf_champ_total`) are excluded here, since they do not exist
until the season is nearly over.

In [ ]:
# Refit at successively later cutoffs
weekly_features = [c for c in feature_cols if not c.startswith('nqa_')]
cutoffs = [((1, 15), 'Jan 15'), ((2, 1), 'Feb 1'), ((2, 15), 'Feb 15'), ((3, 1), 'Mar 1'),
           ((3, 15), 'Mar 15'), ((4, 1), 'Apr 1'), ((4, 8), 'Apr 8')]

track  = ['Stanford', 'Oklahoma', 'Michigan', 'Nebraska']
colors = {'Stanford': '#8C1515', 'Oklahoma': '#841617',
          'Michigan': '#00274C', 'Nebraska': '#E41C38'}
styles = {'Stanford': '-', 'Oklahoma': '--', 'Michigan': '-', 'Nebraska': '--'}

def build(records, cm, cd, year_of=None):
    out = []
    for rec in records:
        f = compute_features(rec, cutoff_month=cm, cutoff_day=cd, min_meets=2)
        if f is None:
            continue
        f['year']     = year_of or rec['year']
        f['team']     = rec['team_name']
        f['champion'] = rec.get('champion', 0)
        out.append(f)
    return pd.DataFrame(out)

def zscore(frame, cols, by_season=True):
    z = frame.copy()
    for c in cols:
        if by_season:
            mu = frame.groupby('year')[c].transform('mean')
            sd = frame.groupby('year')[c].transform('std').replace(0, 1)
        else:
            mu, sd = frame[c].mean(), (frame[c].std() or 1)
        z[c] = (frame[c] - mu) / sd
    return z

by_cutoff = []
for (cm, cd), label in cutoffs:
    hist = build(raw_data, cm, cd)
    if hist.empty:
        continue
    cols = [c for c in weekly_features if c in hist.columns]
    hz   = zscore(hist, cols)

    tr = hz[hz['year'] <= 2023]
    if tr['champion'].sum() == 0:
        print(f"{label}: skipped, no champions have enough meets yet")
        continue

    m = LogisticRegression(C=0.1, class_weight='balanced',
                           max_iter=1000, solver='lbfgs', random_state=42)
    m.fit(tr[cols], tr['champion'])

    entry = {'cutoff': label}
    te = hz[hz['year'] >= 2024].reset_index(drop=True)
    te['prob'] = m.predict_proba(te[cols])[:, 1]
    for yr in (2024, 2025):
        sub = te[te['year'] == yr]
        if sub.empty:
            continue
        champ = sub.loc[sub['champion'] == 1, 'team'].iloc[0]
        entry[f'{yr}_correct'] = sub.loc[sub['prob'].idxmax(), 'team'] == champ
        for t in track:
            row = sub[sub['team'] == t]
            if not row.empty:
                entry[f'{yr}_{t}'] = row['prob'].values[0]

    fut = build(raw_2026, cm, cd, year_of=2026)
    if not fut.empty:
        fz = zscore(fut, cols, by_season=False).reset_index(drop=True)
        fz['prob'] = m.predict_proba(fz[cols])[:, 1]
        for t in track:
            row = fz[fz['team'] == t]
            if not row.empty:
                entry[f'2026_{t}'] = row['prob'].values[0]

    by_cutoff.append(entry)
    print(f"{label}: 2024 {'OK' if entry.get('2024_correct') else 'X '}   "
          f"2025 {'OK' if entry.get('2025_correct') else 'X '}")

In [ ]:
# Plot the trajectories
panels = [(2024, 'Stanford'), (2025, 'Michigan'), (2026, 'Stanford')]
fig, axes = plt.subplots(3, 1, figsize=(8, 11), sharex=True)

for ax, (yr, winner) in zip(axes, panels):
    for t in track:
        xs = [e['cutoff'] for e in by_cutoff if f'{yr}_{t}' in e]
        ys = [e[f'{yr}_{t}'] for e in by_cutoff if f'{yr}_{t}' in e]
        if not xs:
            continue
        ax.plot(xs, ys, marker='o', markersize=5, linewidth=2.2,
                linestyle=styles[t], color=colors[t],
                label=t + (" (won)" if t == winner else ""))

    if yr < 2026:
        for e in by_cutoff:
            if f'{yr}_correct' in e:
                ax.annotate('OK' if e[f'{yr}_correct'] else 'X',
                            (e['cutoff'], 0.02), ha='center', fontsize=8,
                            color='#2a7' if e[f'{yr}_correct'] else '#c33')

    ax.set_title(f"{yr}: {winner} won", loc='left', fontsize=11, fontweight='bold')
    ax.set_ylabel("P(champion)")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.legend(frameon=False, fontsize=8, loc='upper left')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

axes[-1].set_xlabel("Data cutoff")
fig.suptitle("How the champion probability evolves through a season",
             fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

The picture is unstable until roughly **March 1**, and the Apr 1 cutoff is *worse* than Mar 15,
which is a useful warning rather than a bug. Late-season meets are sparse and high-variance, so a
cutoff that lands mid-conference-season catches teams with one or two unrepresentative results. The
value of April 8 is not that later is always better; it is that April 8 sits after conference
championships have resolved, which is what stabilizes the estimate.

With 12 champions total, these trajectories should be read as suggestive, not measured.

## 16. Conclusions and Limitations

### What worked

**Feature engineering mattered more than model selection.** Every gain we got came from building
better inputs, not from fitting better models. Swapping per-event means for per-event NQA moved the
parallel-bars baseline from 10/12 to 12/12. Within-season z-scoring is the only reason a model
trained on 2013 scoring rules can say anything about 2026. Neither needed anything fancier than
logistic regression.

**Parallel bars is where the signal is.** Two independent routes agree on this. The regression put
`nqa_pb` and `max_pb` in its top two weights, and when we followed that hint and tested p-bars
alone, it separated champions better than any whole-team measure we tried.

**The model earned its place as a search tool.** It did not beat the single-feature rule, but we
found the single-feature rule by reading its weights. That is a real contribution even though the
classifier is not the deliverable.

**The data integrity work paid off.** The conference filter that resolved the 6.6% flag rate is the
same filter that keeps GymACT seasons out of the training data.

### What did not

**The model could not separate the top two.** On 2026 it narrowed the field to the right four teams
and then split hairs between Stanford and Oklahoma, with different versions landing on different
answers. At a 1.33-point margin we do not think any of them constitutes a real call.

**The SVM produced confident nonsense.** Correct top picks, incoherent rankings underneath.

### Limitations

1. **The sample is tiny.** 12 labeled seasons, 12 champions, 3 distinct winning programs. The 12/12
   baseline was selected after inspecting these seasons, and would flatter itself on any resample.
   One genuine out-of-sample season (2026) is not enough to settle it.

2. **Three programs win everything.** Michigan, Oklahoma, and Stanford took every title from 2013 to
   2026. A model that learned "be Stanford" would score well, and there is no clean way to rule that
   out from within this dataset. Extending training back to 2013-2015 was tried specifically to
   dilute the Stanford preference; it did not remove it.

3. **`mean_total` may be misaligned.** The season-level totals come from the `dashboard` endpoint
   while the apparatus scores and dates come from `teamconsistency`, and the two do not always
   contain the same number of meets. `compute_features` takes `meet_scores[:len(valid)]`, a
   positional slice that assumes the two lists agree from the start. The apparatus features are
   unaffected, since they index `teamconsistency` directly, but the four `*_total` features should be
   re-derived by joining on date before being trusted.

4. **NQA degrades on teams with very few meets.** The formula takes `sorted_reg[1:4]` and divides
   by 4, so a team-season with fewer than five pre-cutoff meets contributes fewer than three
   regular-season scores to a sum that is still divided by four, deflating its NQA. At the April 8
   cutoff most teams have 8-12 meets so this rarely binds, but it is a latent trap for short seasons
   and for the early cutoffs in Section 15. Worth adding an explicit guard.

5. **The April 8 cutoff is asserted, not proven.** It should be re-verified against the meet labels
   in every season (see the TODO in Section 6).

6. **NQA was never checked against RTN's published figures.** The implementation here matches the
   described NCAA procedure and produces sensible results, but it has not been reconciled against
   RTN's own NQA/RQS numbers. The 13/13 agreement with champions is circumstantial support, not
   verification.

7. **No confidence intervals anywhere.** With 12 positive examples, the difference between 10/12 and
   12/12 is one or two seasons, well inside the noise.

### If this continued

- Reconcile the NQA implementation against RTN's published values.
- Re-derive the `*_total` features with a date join instead of a positional slice.
- Test whether p-bars dominance is causal or an artifact of the same programs leading everything,
  for example by checking whether `nqa_pb` still separates champions once overall team strength is
  partialled out.
- Predict the **top four** rather than the winner. The forward test suggests the model is
  meaningfully better at bracketing contenders than at ordering them, and a four-team target would
  have 4x the positive examples.
- Topological data analysis was discussed as a direction and deferred for time.

---

*Data: [Road to Nationals](https://www.roadtonationals.com) men's gymnastics API.
Seasons 2013-2026, 2020 excluded (COVID-19).*